# PIDNet

PIDNet es una arquitectura para segmetación semántica en tiempo real que emplea el concepto básico de un controlado PID (Proportional-Integral-Derivative). Muchos modelos con gran balance de exactitud y velocidad adecuados para segmentación en tiempo real (BiSeNet, Fast-SCNN, DDRNet, etc) son tipo TBN (Two-Branch-Network). Al analizar esta estrategia desde la perspectiva de un controlador PID, se puede hacer una equivalencia de dicha arquitectura con un controlador PI, el cuál suele tener problemas de sobrepaso (overshoot). A nivel imagen, esto puede traducirse en una rama que aprovecha la información semántica original (P) y otra rama que almacena información contextual de baja frecuencia (I), tal que un modelo de segmentación que usa la fusión de ambas conlleva el riesgo de que los límites de los objetos se vean excesivamente erosionados por los pixeles circundantes y que los objetos pequeños quede eclipsados por los objetos grandes adyacentes.

Para mitigar eso, se propone adjuntar una tercera rama "derivativa". La rama derivativa de un controlador PID se enfoca en la velocidad de cambio de una señal, permitiendo una mejor sensibilidad a cambios de alta frecuencia. En una imagen, esto representa un mejor enfoque sobre los bordes de los objetos o regiones. Dado que las grietas suelen tener una elevada relación perímetro-área, una red cuya detección de bordes está mejorada, en teoría resulta ideal para una aplicación de segmentación de grietas.

La red propuesta, además utiliza los siguientes módulos específicos para que las ramas interactúen de forma inteligente:
- Pag (Pixel-attention-guided fusion): Este módulo permite que la rama de detalles (P) aprenda selectivamente características semánticas ricas de la rama de contexto (I) sin ser abrumada por ella. Utiliza un mecanismo de atención por píxel para decidir cuánta confianza otorgar a la información de contexto en cada punto.
- PAPPM (Parallel Aggregation Pyramid Pooling Module): Es una versión optimizada del módulo PPM tradicional. En lugar de procesar las escalas de forma secuencial, lo hace de forma paralela para reducir la latencia y mantener la velocidad necesaria en aplicaciones de tiempo real.
- Bag (Boundary-attention-guided fusion): Es el corazón de la integración PID. Utiliza las fronteras detectadas por la rama D para guiar la fusión entre la rama de detalles (P) y la de contexto (I). Básicamente, "obliga" a la red a confiar más en la rama de detalles cuando está cerca de un borde y en la de contexto cuando está dentro de un objeto.

La función de pérdida de la red resulta más compleja. Se compone una suma ponderada de 4 pérdidas que balancean el aprendizaje de bordes y semántica.

$$ Loss = \lambda_0l_0 + \lambda_1l_1 + \lambda_2l_2 + \lambda_3l_3$$
1. $l0$: Pérdida semántica auxiliar para optimizar toda la red
2. $l1$: Pérdida de entropía cruzada binaria ponderada para la detección de bordes (rama D)
3. $l2$: Pérdida de entropía cruzada estándar para la segmentación final
4. $l3$: Pérdida de entropía cruzada con conciencia de frontera (boundary-awareness), que coordina las tareas de segmentación y detección de bordes

**Adaptación para rama derivativa**: La rama derivativa requiere una supervisión directa mediante mapas de bordes reales para entrenarse correctamente. Dado que el dataset de DeepCrack proporciona únicamente las máscaras binarias grieta/fondo, es necesario generar también las etiquetas de borde. Para ello, se propone generarlas usando un detector de bordes de Canny durante la misma carga del dataset (on-the-fly).

In [ ]:
# ==== Solo para ejecutar en Colab / Local ========
try:
    from google.colab import drive
    import os
    import sys

    # 1. Montar Google Drive
    drive.mount('/content/drive')

    # 2. Definir la ruta a tu proyecto dentro de Drive
    project_path = '/content/drive/MyDrive/tp_computer_vision_ii'
    notebook_dir = os.path.join(project_path, 'src', 'pidnet')

    # 3. Movernos al directorio donde está el notebook para que las rutas relativas funcionen
    os.chdir(notebook_dir)

    # 4. Agregar la ruta a sys.path para que Python encuentre la carpeta "models"
    if notebook_dir not in sys.path:
        sys.path.append(notebook_dir)

    print("Directorio actual:", os.getcwd())
    IN_COLAB = True

except:
    IN_COLAB = False
    import os, sys
    current_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in locals() else os.getcwd()
    if current_dir not in sys.path:
        sys.path.append(current_dir)


In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
import random
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from pathlib import Path

from models.pidnet import PIDNet

# Configuración de dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de cómputo seleccionado: {device}")

# Función para fijar semillas y garantizar reproducibilidad experimental
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)


In [ ]:
# Constantes estándar de normalización ImageNet
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def unnormalize_image(img_tensor):
    """
    Convierte un tensor de imagen normalizado [C, H, W] o arreglo [H, W, C] 
    de vuelta al espacio RGB [0, 255] uint8 para visualización fiel sin distorsión de color.
    """
    if isinstance(img_tensor, torch.Tensor):
        img_np = img_tensor.detach().cpu().permute(1, 2, 0).numpy()
    else:
        img_np = img_tensor.copy()
    img_np = (img_np * IMAGENET_STD + IMAGENET_MEAN) * 255.0
    return np.clip(img_np, 0, 255).astype(np.uint8)


class JointTransform:
    """
    Aplica transformaciones geométricas y fotométricas simultáneas a la imagen y máscara.
    Optimizado para segmentación de grietas (invariante a escala, orientación y flip).
    """
    def __init__(
        self, 
        scale_limit=(0.5, 2.0),
        crop_size=(512, 512),
        hflip_p=0.5,
        vflip_p=0.5,
        max_rotate_deg=10,
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ):
        self.scale_limit = scale_limit
        self.crop_h, self.crop_w = crop_size
        self.hflip_p = hflip_p
        self.vflip_p = vflip_p
        self.max_rotate_deg = max_rotate_deg
        self.mean = np.array(mean, dtype=np.float32)
        self.std = np.array(std, dtype=np.float32)

    def __call__(self, image, mask):
        # 1. Escalado aleatorio (Random Scale)
        scale = random.uniform(*self.scale_limit)
        h, w = image.shape[:2]
        new_h, new_w = max(1, int(h * scale)), max(1, int(w * scale))
        image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)

        # 2. Relleno (Padding si tras escalar queda menor que crop_size)
        pad_h = max(self.crop_h - new_h, 0)
        pad_w = max(self.crop_w - new_w, 0)
        if pad_h > 0 or pad_w > 0:
            image = cv2.copyMakeBorder(image, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)
            mask = cv2.copyMakeBorder(mask, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)

        # 3. Recorte aleatorio (Random Crop)
        h, w = image.shape[:2]
        top = random.randint(0, h - self.crop_h)
        left = random.randint(0, w - self.crop_w)
        image = image[top:top + self.crop_h, left:left + self.crop_w]
        mask = mask[top:top + self.crop_h, left:left + self.crop_w]

        # 4. Volteos aleatorios (Horizontal y Vertical Flips)
        if random.random() < self.hflip_p:
            image = cv2.flip(image, 1)
            mask = cv2.flip(mask, 1)
        if random.random() < self.vflip_p:
            image = cv2.flip(image, 0)
            mask = cv2.flip(mask, 0)

        # 5. Rotación aleatoria leve
        if self.max_rotate_deg > 0 and random.random() < 0.5:
            angle = random.uniform(-self.max_rotate_deg, self.max_rotate_deg)
            ch, cw = image.shape[:2]
            M = cv2.getRotationMatrix2D((cw / 2, ch / 2), angle, 1.0)
            image = cv2.warpAffine(image, M, (cw, ch), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
            mask = cv2.warpAffine(mask, M, (cw, ch), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)

        # 6. Variación fotométrica leve (Brillo/Contraste solo sobre la imagen)
        if random.random() < 0.5:
            alpha = random.uniform(0.8, 1.2)
            beta = random.uniform(-20, 20)
            image = np.clip(image.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

        return image, mask

    def normalize_and_to_tensor(self, image):
        image = image.astype(np.float32) / 255.0
        image = (image - self.mean) / self.std
        return torch.from_numpy(image).permute(2, 0, 1).float()


class ValTransform:
    """
    Transformación determinista para validación y prueba: redimensionado y normalización ImageNet.
    """
    def __init__(self, target_size=(512, 512), mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
        self.target_size = target_size
        self.mean = np.array(mean, dtype=np.float32)
        self.std  = np.array(std,  dtype=np.float32)

    def __call__(self, image, mask):
        if self.target_size is not None:
            image = cv2.resize(image, (self.target_size[1], self.target_size[0]), interpolation=cv2.INTER_LINEAR)
            mask = cv2.resize(mask, (self.target_size[1], self.target_size[0]), interpolation=cv2.INTER_NEAREST)
        return image, mask

    def normalize_and_to_tensor(self, image):
        image = image.astype(np.float32) / 255.0
        image = (image - self.mean) / self.std
        return torch.from_numpy(image).permute(2, 0, 1).float()


In [ ]:
class DeepCrackDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None, target_size=(512, 512)):
        """
        split: 'train' o 'test'
        """
        self.root_dir = root_dir
        self.img_dir = os.path.join(self.root_dir, f"{split}_img")
        self.lab_dir = os.path.join(self.root_dir, f"{split}_lab")
        if not os.path.exists(self.img_dir):
            raise FileNotFoundError(f"No se encontró el directorio de imágenes: {self.img_dir}")
        self.images = sorted(os.listdir(self.img_dir))
        self.transform = transform
        self.target_size = target_size
        self.mean = np.array((0.485, 0.456, 0.406), dtype=np.float32)
        self.std = np.array((0.229, 0.224, 0.225), dtype=np.float32)

    def __len__(self):
        return len(self.images)

    def _generate_boundary_on_the_fly(self, mask):
        """
        Genera el mapa de bordes binario mediante Canny y dilatación morfológica 
        para la supervisión directa de la rama derivativa (D) de PIDNet.
        """
        if mask.max() == 1:
            mask_uint8 = (mask * 255).astype(np.uint8)
        else:
            mask_uint8 = mask.astype(np.uint8)

        edges = cv2.Canny(mask_uint8, 10, 100)
        kernel = np.ones((3, 3), np.uint8)
        dilated_edges = cv2.dilate(edges, kernel, iterations=1)
        return (dilated_edges > 0).astype(np.float32)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        lab_name = os.path.splitext(img_name)[0] + '.png'
        lab_path = os.path.join(self.lab_dir, lab_name)

        # Cargar imagen y máscara
        image = cv2.imread(img_path, cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(lab_path, cv2.IMREAD_GRAYSCALE)

        # Binarizar máscara
        mask = (mask > 127).astype(np.uint8)

        if self.transform is not None:
            image, mask = self.transform(image, mask)
        else:
            if self.target_size is not None:
                image = cv2.resize(image, (self.target_size[1], self.target_size[0]), interpolation=cv2.INTER_LINEAR)
                mask = cv2.resize(mask, (self.target_size[1], self.target_size[0]), interpolation=cv2.INTER_NEAREST)

        # Generar etiquetas de frontera on-the-fly para la rama derivativa
        boundary = self._generate_boundary_on_the_fly(mask)

        # Normalización a tensor
        if hasattr(self.transform, 'normalize_and_to_tensor'):
            image_tensor = self.transform.normalize_and_to_tensor(image)
        else:
            norm_img = (image.astype(np.float32) / 255.0 - self.mean) / self.std
            image_tensor = torch.from_numpy(norm_img).permute(2, 0, 1).float()

        mask_tensor = torch.from_numpy(mask.astype(np.int64))
        boundary_tensor = torch.from_numpy(boundary)

        return image_tensor, mask_tensor, boundary_tensor


### Inspección Visual del Dataset y Generación de Bordes (Canny)
A continuación se visualizan **2 juegos de 4 imágenes** tomadas aleatoriamente del dataset para verificar el correcto funcionamiento del pipeline de carga, aumentación y extracción de bordes on-the-fly:
1. **Imagen Original** (RGB des-normalizada).
2. **Máscara Ground Truth** (Grieta binaria).
3. **Bordes Canny con dilatación** (Supervisión de la rama derivativa D de PIDNet).
4. **Superposición Completa** (Imagen + Máscara roja semitransparente + Contornos de borde amarillos).


In [ ]:
def plot_dataset_samples_with_edges(dataset, num_samples=2, figsize=(18, 9)):
    """
    Genera una figura con `num_samples` filas x 4 columnas conteniendo:
    1. Imagen original (RGB des-normalizada)
    2. Máscara Ground Truth (Grieta)
    3. Bordes generados por Canny (Supervisión Rama D)
    4. Superposición completa (Imagen + Máscara coloreada + Bordes Canny resaltados)
    """
    indices = random.sample(range(len(dataset)), num_samples)
    fig, axes = plt.subplots(num_samples, 4, figsize=figsize)
    
    col_titles = [
        '1. Imagen Original (RGB)',
        '2. Máscara Ground Truth',
        '3. Bordes Canny (Rama D)',
        '4. Superposición Completa'
    ]
    
    for r_idx, s_idx in enumerate(indices):
        img_tensor, mask_tensor, boundary_tensor = dataset[s_idx]
        
        # 1. Imagen RGB des-normalizada
        img_rgb = unnormalize_image(img_tensor)
        
        # 2. Máscara GT [H, W]
        mask_np = mask_tensor.cpu().numpy().astype(np.uint8)
        
        # 3. Bordes Canny [H, W]
        boundary_np = boundary_tensor.cpu().numpy().astype(np.uint8)
        
        # 4. Superposición (Overlay)
        overlay = img_rgb.copy()
        mask_color = np.zeros_like(img_rgb)
        mask_color[mask_np == 1] = [255, 50, 50]  # Grieta en rojo
        alpha_mask = 0.4
        overlay = np.where(
            mask_np[..., None] == 1, 
            cv2.addWeighted(overlay, 1 - alpha_mask, mask_color, alpha_mask, 0), 
            overlay
        )
        # Contorno Canny en amarillo brillante
        overlay[boundary_np == 1] = [255, 255, 0]
        
        # Graficar
        axes[r_idx, 0].imshow(img_rgb)
        axes[r_idx, 0].set_ylabel(f'Muestra #{s_idx}', fontsize=12, fontweight='bold')
        
        axes[r_idx, 1].imshow(mask_np, cmap='gray', vmin=0, vmax=1)
        axes[r_idx, 2].imshow(boundary_np, cmap='magma', vmin=0, vmax=1)
        axes[r_idx, 3].imshow(overlay)
        
        for c_idx in range(4):
            if r_idx == 0:
                axes[r_idx, c_idx].set_title(col_titles[c_idx], fontsize=13, fontweight='bold', pad=10)
            axes[r_idx, c_idx].axis('off')

    plt.tight_layout()
    plt.show()

# Instanciar dataset temporal para visualización y graficar 2 muestras aleatorias
sample_dataset = DeepCrackDataset(root_dir='../../datasets/dataset_1000', split='train', transform=JointTransform())
plot_dataset_samples_with_edges(sample_dataset, num_samples=2)


In [ ]:
def build_custom_pidnet(pretrained_path='pretrained_models/PIDNet_S_Cityscapes_test.pt', num_classes=2, variant='S'):
    """
    Construye la arquitectura PIDNet (S, M, o L) y carga los pesos pre-entrenados de Cityscapes,
    adaptando automáticamente las cabezas de segmentación (`final_layer` y `seghead_p`) al número
    de clases del dataset objetivo (2 para DeepCrack), y conservando `seghead_d` (1 canal).
    """
    if variant == 'S':
        model = PIDNet(m=2, n=3, num_classes=num_classes, planes=32, ppm_planes=96, head_planes=128, augment=True)
    elif variant == 'M':
        model = PIDNet(m=2, n=3, num_classes=num_classes, planes=64, ppm_planes=96, head_planes=128, augment=True)
    elif variant == 'L':
        model = PIDNet(m=3, n=4, num_classes=num_classes, planes=64, ppm_planes=112, head_planes=256, augment=True)
    else:
        raise ValueError(f"Variante '{variant}' no soportada. Opciones válidas: 'S', 'M', 'L'")

    if os.path.exists(pretrained_path):
        print(f"Cargando pesos pre-entrenados desde: {pretrained_path}")
        pretrained_dict = torch.load(pretrained_path, map_location='cpu')

        if 'state_dict' in pretrained_dict:
            pretrained_dict = pretrained_dict['state_dict']
        pretrained_dict = {k.replace('module.', '').replace('model.', ''): v for k, v in pretrained_dict.items()}

        model_dict = model.state_dict()
        filtered_dict = {
            k: v for k, v in pretrained_dict.items()
            if k in model_dict and v.shape == model_dict[k].shape
        }

        model_dict.update(filtered_dict)
        model.load_state_dict(model_dict)

        omitted = set(pretrained_dict.keys()) - set(filtered_dict.keys())
        print(f"--- Diagnóstico de Carga ---")
        print(f"Capas pre-entrenadas cargadas exitosamente: {len(filtered_dict)}")
        print(f"Capas adaptadas/reinicializadas para {num_classes} clases: {len(omitted)}")
        for k in sorted(omitted):
            print(f" -> {k} (Forma original: {pretrained_dict[k].shape})")
    else:
        print(f"[ADVERTENCIA] No se encontró el archivo {pretrained_path}. El modelo se inicializa aleatoriamente.")

    return model


In [ ]:
NUM_CLASSES = 2
VARIANT = "S"
pretrained_path = os.path.join('pretrained_models', f'PIDNet_{VARIANT}_Cityscapes_test.pt')

model = build_custom_pidnet(
    pretrained_path=pretrained_path,
    num_classes=NUM_CLASSES,
    variant=VARIANT
)


In [ ]:
model


In [ ]:
# Configuración de num_workers según entorno y sistema operativo
num_workers = 2 if (device.type == 'cuda' and os.name != 'nt') else 0

train_transform = JointTransform(
    scale_limit=(0.5, 2.0), 
    crop_size=(512, 512), 
    hflip_p=0.5, 
    vflip_p=0.5, 
    max_rotate_deg=10
)
val_transform = ValTransform(target_size=(512, 512))

# Instanciar Datasets y DataLoaders
train_dataset = DeepCrackDataset(root_dir='../../datasets/dataset_1000', split='train', transform=train_transform)
train_loader = DataLoader(
    train_dataset, 
    batch_size=8, 
    shuffle=True, 
    num_workers=num_workers, 
    pin_memory=(device.type == 'cuda'),
    drop_last=True
)

val_dataset = DeepCrackDataset(root_dir='../../datasets/dataset_1000', split='test', transform=val_transform)
val_loader = DataLoader(
    val_dataset, 
    batch_size=8, 
    shuffle=False, 
    num_workers=num_workers, 
    pin_memory=(device.type == 'cuda')
)

print(f"Train Dataset: {len(train_dataset)} imágenes ({len(train_loader)} batches)")
print(f"Val Dataset:   {len(val_dataset)} imágenes ({len(val_loader)} batches)")


In [ ]:
class BoundaryAwareCrossEntropy(nn.Module):
    def __init__(self, gamma=1.0, ignore_index=255):
        super(BoundaryAwareCrossEntropy, self).__init__()
        self.gamma = gamma
        self.ignore_index = ignore_index

    def forward(self, semantic_preds, boundary_preds, targets):
        """
        semantic_preds: Logits de predicción semántica [B, C, H, W]
        boundary_preds: Probabilidades Sigmoide de la rama de bordes [B, 1, H, W]
        targets: Etiquetas ground truth [B, H, W]
        """
        ce_loss = F.cross_entropy(semantic_preds, targets, reduction='none', ignore_index=self.ignore_index)
        
        # Ponderación espacial según predicción de frontera: (1 + gamma * p_boundary)
        boundary_weight = 1.0 + (self.gamma * boundary_preds.squeeze(1))
        weighted_ce = ce_loss * boundary_weight

        # Promedio únicamente sobre píxeles válidos
        valid_mask = (targets != self.ignore_index).float()
        valid_pixels = valid_mask.sum().clamp(min=1.0)
        return (weighted_ce * valid_mask).sum() / valid_pixels


In [ ]:
# 1. Definición de las funciones de pérdida
# l_0 y l_2: Pérdida semántica auxiliar y principal
criterion_semantic = nn.CrossEntropyLoss(ignore_index=255)

# l_1: Weighted BCE para la rama de bordes (pos_weight compensa la extrema escasez de píxeles de borde)
boundary_pos_weight = torch.tensor([15.0]).to(device)
criterion_boundary = nn.BCEWithLogitsLoss(pos_weight=boundary_pos_weight)

# l_3: Pérdida con conciencia de frontera (Boundary-Aware CE)
criterion_boundary_aware = BoundaryAwareCrossEntropy(gamma=1.0, ignore_index=255)

# 2. Optimizador (SGD con Nesterov y Momentum según paper oficial)
optimizer = optim.SGD(
    model.parameters(),
    lr=1e-3,
    momentum=0.9,
    weight_decay=5e-4,
    nesterov=True
)

# 3. Poly Learning Rate Scheduler
max_epochs = 50
power = 0.9
def poly_lr_scheduler(epoch):
    return (1 - epoch / max_epochs) ** power

scheduler = LambdaLR(optimizer, lr_lambda=poly_lr_scheduler)


In [ ]:
def compute_conf_matrix(preds, targets, num_classes=2, ignore_index=255):
    """
    Calcula la matriz de confusión acumulada [num_classes, num_classes] (fila=real, col=predicho).
    """
    pred_labels = preds.argmax(dim=1).view(-1)
    target_flat = targets.view(-1)

    valid = target_flat != ignore_index
    pred_flat = pred_labels[valid]
    target_flat = target_flat[valid]

    indices = num_classes * target_flat + pred_flat
    conf_matrix = torch.bincount(indices, minlength=num_classes ** 2)
    return conf_matrix.reshape(num_classes, num_classes)


def compute_metrics(conf_matrix):
    """
    Calcula métricas detalladas: mIoU, IoU por clase, Precision, Recall y F1-Score (Dice).
    Especialmente relevante para monitorear el desempeño específico sobre la clase 'grieta'.
    """
    tp = conf_matrix.diag().float()
    fp = (conf_matrix.sum(dim=0) - tp).float()
    fn = (conf_matrix.sum(dim=1) - tp).float()
    denom = tp + fp + fn

    valid_classes = denom > 0
    iou_per_class = torch.zeros_like(tp)
    iou_per_class[valid_classes] = tp[valid_classes] / denom[valid_classes].clamp(min=1e-7)

    precision = tp / (tp + fp).clamp(min=1e-7)
    recall = tp / (tp + fn).clamp(min=1e-7)
    f1 = 2 * (precision * recall) / (precision + recall).clamp(min=1e-7)

    miou = iou_per_class[valid_classes].mean().item()
    bg_iou = iou_per_class[0].item() if len(iou_per_class) > 0 else 0.0
    crack_iou = iou_per_class[1].item() if len(iou_per_class) > 1 else 0.0
    crack_f1 = f1[1].item() if len(f1) > 1 else 0.0
    crack_prec = precision[1].item() if len(precision) > 1 else 0.0
    crack_rec = recall[1].item() if len(recall) > 1 else 0.0

    return {
        'miou': miou,
        'bg_iou': bg_iou,
        'crack_iou': crack_iou,
        'crack_f1': crack_f1,
        'crack_precision': crack_prec,
        'crack_recall': crack_rec,
        'iou_per_class': iou_per_class.tolist()
    }


def run_epoch(
    model, loader, device, num_classes,
    criterion_semantic, criterion_boundary, criterion_boundary_aware,
    optimizer=None, scaler=None
):
    """
    Ejecuta un epoch completo de entrenamiento o evaluación con soporte para Automatic Mixed Precision (AMP).
    Retorna (loss_promedio, metrics_dict).
    """
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    conf_matrix = torch.zeros(num_classes, num_classes, dtype=torch.long, device=device)
    use_amp = (device.type == 'cuda')

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for images, masks, boundaries in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            boundaries = boundaries.to(device, non_blocking=True).unsqueeze(1) # [B, 1, H, W]

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', enabled=use_amp):
                outputs = model(images)
                pred_p = outputs[0]
                pred_final = outputs[1]
                pred_d = outputs[2]

                # Interpolar al tamaño del target
                h, w = masks.shape[1], masks.shape[2]
                pred_p     = F.interpolate(pred_p,     size=(h, w), mode='bilinear', align_corners=True)
                pred_final = F.interpolate(pred_final, size=(h, w), mode='bilinear', align_corners=True)
                pred_d     = F.interpolate(pred_d,     size=(h, w), mode='bilinear', align_corners=True)

                loss_0 = criterion_semantic(pred_p, masks)
                loss_1 = criterion_boundary(pred_d, boundaries)
                loss_2 = criterion_semantic(pred_final, masks)
                pred_d_sigmoid = torch.sigmoid(pred_d)
                loss_3 = criterion_boundary_aware(pred_final, pred_d_sigmoid, masks)

                # Coeficientes oficiales del paper PIDNet
                loss_total = (0.4 * loss_0) + (20.0 * loss_1) + (1.0 * loss_2) + (1.0 * loss_3)

            if is_train:
                if scaler is not None and use_amp:
                    scaler.scale(loss_total).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss_total.backward()
                    optimizer.step()

            total_loss += loss_total.item()
            conf_matrix += compute_conf_matrix(pred_final.detach(), masks, num_classes)

    avg_loss = total_loss / len(loader)
    metrics = compute_metrics(conf_matrix)
    return avg_loss, metrics


In [ ]:
model.to(device)

# Historial para análisis y curvas
history = {
    'train_loss': [],
    'train_miou': [],
    'train_crack_iou': [],
    'train_crack_f1': [],
    'val_loss': [],
    'val_miou': [],
    'val_crack_iou': [],
    'val_crack_f1': [],
    'learning_rates': []
}

best_val_crack_iou = 0.0
patience = 12
patience_counter = 0

# GradScaler para entrenamiento con Mixed Precision (AMP)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

save_checkpoint_name = f'best_pidnet_{VARIANT}.pth'

print("================ INICIANDO ENTRENAMIENTO PIDNet ================")
for epoch in range(max_epochs):

    # ======= ENTRENAMIENTO =======
    train_loss, train_metrics = run_epoch(
        model, train_loader, device, NUM_CLASSES,
        criterion_semantic, criterion_boundary, criterion_boundary_aware,
        optimizer=optimizer, scaler=scaler
    )

    # ===== VALIDACIÓN ======
    val_loss, val_metrics = run_epoch(
        model, val_loader, device, NUM_CLASSES,
        criterion_semantic, criterion_boundary, criterion_boundary_aware,
        optimizer=None
    )

    # ====== GUARDAR HISTORIALES ======
    history['train_loss'].append(train_loss)
    history['train_miou'].append(train_metrics['miou'])
    history['train_crack_iou'].append(train_metrics['crack_iou'])
    history['train_crack_f1'].append(train_metrics['crack_f1'])

    history['val_loss'].append(val_loss)
    history['val_miou'].append(val_metrics['miou'])
    history['val_crack_iou'].append(val_metrics['crack_iou'])
    history['val_crack_f1'].append(val_metrics['crack_f1'])
    
    current_lr = optimizer.param_groups[0]['lr']
    history['learning_rates'].append(current_lr)

    # ========== LOGGING ==========
    print(f"--- Epoch {epoch+1:02d}/{max_epochs:02d} | LR: {current_lr:.6f} ---")
    print(f"Train → Loss: {train_loss:.4f} | mIoU: {train_metrics['miou']:.4f} | Crack IoU: {train_metrics['crack_iou']:.4f} | F1: {train_metrics['crack_f1']:.4f}")
    print(f"Val   → Loss: {val_loss:.4f}   | mIoU: {val_metrics['miou']:.4f} | Crack IoU: {val_metrics['crack_iou']:.4f} | F1: {val_metrics['crack_f1']:.4f}")
    print(f"IoU por clase (val): Fondo={val_metrics['bg_iou']:.4f}, Grieta={val_metrics['crack_iou']:.4f}")

    # ====== GUARDAR MEJOR MODELO (Monitoreando Crack IoU) ======
    if val_metrics['crack_iou'] > best_val_crack_iou:
        best_val_crack_iou = val_metrics['crack_iou']
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_metrics': val_metrics,
            'val_loss': val_loss,
            'variant': VARIANT,
            'num_classes': NUM_CLASSES
        }, save_checkpoint_name)
        print(f"  -> ¡Nuevo mejor modelo guardado en {save_checkpoint_name}! (Val Crack IoU: {best_val_crack_iou:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1

    # ====== EARLY STOPPING ======
    if patience_counter >= patience:
        print(f"\n[INFO] Early Stopping activado tras {patience} épocas sin mejora en la época {epoch + 1}")
        break

    # ====== ACTUALIZAR LEARNING RATE ======
    scheduler.step()
    print()

print("================ ENTRENAMIENTO FINALIZADO ================")


In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Pérdida (Loss)
axes[0].plot(epochs_range, history['train_loss'], label='Train Loss', color='royalblue', lw=2)
axes[0].plot(epochs_range, history['val_loss'], label='Val Loss', color='crimson', lw=2)
axes[0].set_xlabel('Época', fontsize=11)
axes[0].set_ylabel('Loss Compuesta', fontsize=11)
axes[0].set_title('Evolución de Pérdida (Train vs Val)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. IoU de Grieta & mIoU
axes[1].plot(epochs_range, history['train_crack_iou'], label='Train Crack IoU', color='darkorange', lw=2)
axes[1].plot(epochs_range, history['val_crack_iou'], label='Val Crack IoU', color='seagreen', lw=2)
axes[1].plot(epochs_range, history['val_miou'], label='Val mIoU (Global)', color='gray', lw=1.5, linestyle='--')
axes[1].set_xlabel('Época', fontsize=11)
axes[1].set_ylabel('IoU', fontsize=11)
axes[1].set_title('Evolución de IoU (Grieta & Global)', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. F1-Score (Dice)
axes[2].plot(epochs_range, history['train_crack_f1'], label='Train F1 (Grieta)', color='purple', lw=2)
axes[2].plot(epochs_range, history['val_crack_f1'], label='Val F1 (Grieta)', color='teal', lw=2)
axes[2].set_xlabel('Época', fontsize=11)
axes[2].set_ylabel('F1-Score / Dice', fontsize=11)
axes[2].set_title('Evolución de F1-Score (Grieta)', fontsize=12, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Resumen final
best_epoch = history['val_crack_iou'].index(max(history['val_crack_iou'])) + 1
print(f"\n================ RESUMEN FINAL ================")
print(f"Mejor Época:              {best_epoch}")
print(f"Mejor Val Crack IoU:      {max(history['val_crack_iou']):.4f}")
print(f"Mejor Val F1-Score:       {history['val_crack_f1'][best_epoch-1]:.4f}")
print(f"Val mIoU en mejor época:  {history['val_miou'][best_epoch-1]:.4f}")
print(f"Loss de validación final: {history['val_loss'][-1]:.4f}")


In [ ]:
import random
import time

# ─── Configuración ────────────────────────────────────────────────────────────
CHECKPOINT_PATH = f'best_pidnet_{VARIANT}.pth'
NUM_IMAGES      = 10

# ─── Cargar modelo desde checkpoint ───────────────────────────────────────────
if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    val_m = checkpoint.get('val_metrics', {})
    print(f"Modelo cargado desde {CHECKPOINT_PATH} - Época {checkpoint['epoch']} | Val Crack IoU: {val_m.get('crack_iou', 0.0):.4f}")
else:
    print(f"[ADVERTENCIA] No se encontró {CHECKPOINT_PATH}. Evaluando con el modelo en memoria.")
    model.eval()

# ─── Seleccionar imágenes aleatorias del val/test set ─────────────────────────
indices = random.sample(range(len(val_dataset)), NUM_IMAGES)

fig, axes = plt.subplots(NUM_IMAGES, 4, figsize=(16, 3.8 * NUM_IMAGES))
axes[0, 0].set_title('1. Imagen Original',   fontsize=12, fontweight='bold')
axes[0, 1].set_title('2. Máscara Ground Truth', fontsize=12, fontweight='bold')
axes[0, 2].set_title('3. Predicción PIDNet',    fontsize=12, fontweight='bold')
axes[0, 3].set_title('4. Superposición / Error', fontsize=12, fontweight='bold')

start_time = time.perf_counter()
with torch.no_grad():
    for row, idx in enumerate(indices):
        image_tensor, mask_tensor, _ = val_dataset[idx]

        # Forward pass
        input_tensor = image_tensor.unsqueeze(0).to(device)
        outputs      = model(input_tensor)
        pred_final   = outputs[1] # Salida principal

        h, w       = mask_tensor.shape[0], mask_tensor.shape[1]
        pred_final = F.interpolate(pred_final, size=(h, w), mode='bilinear', align_corners=True)
        pred_mask  = pred_final.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
        mask_np    = mask_tensor.cpu().numpy().astype(np.uint8)

        # Des-normalizar imagen RGB fielmente
        img_rgb = unnormalize_image(image_tensor)

        # Mapa de superposición y errores (Verde: TP, Rojo: FP, Azul: FN)
        overlay = img_rgb.copy()
        tp_mask = (pred_mask == 1) & (mask_np == 1)
        fp_mask = (pred_mask == 1) & (mask_np == 0)
        fn_mask = (pred_mask == 0) & (mask_np == 1)

        overlay[tp_mask] = [0, 255, 0]    # Verde: True Positive
        overlay[fp_mask] = [255, 0, 0]    # Rojo: False Positive
        overlay[fn_mask] = [0, 150, 255]  # Naranja/Azul: False Negative
        overlay_blend = cv2.addWeighted(img_rgb, 0.6, overlay, 0.4, 0)

        # Métricas individuales de la muestra
        tp_count = tp_mask.sum()
        fp_count = fp_mask.sum()
        fn_count = fn_mask.sum()
        denom = tp_count + fp_count + fn_count
        sample_iou = (tp_count / denom) if denom > 0 else 0.0
        sample_f1  = (2 * tp_count / (2 * tp_count + fp_count + fn_count)) if (2 * tp_count + fp_count + fn_count) > 0 else 0.0

        # Plot
        axes[row, 0].imshow(img_rgb)
        axes[row, 1].imshow(mask_np, cmap='gray', vmin=0, vmax=1)
        axes[row, 2].imshow(pred_mask, cmap='gray', vmin=0, vmax=1)
        axes[row, 3].imshow(overlay_blend)

        axes[row, 0].set_ylabel(f'Muestra #{idx}', fontsize=10, fontweight='bold')
        axes[row, 2].set_xlabel(f'IoU: {sample_iou:.3f} | F1: {sample_f1:.3f}', fontsize=10)
        axes[row, 3].set_xlabel('Verde: TP | Rojo: FP | Azul: FN', fontsize=9)

        for ax in axes[row]:
            ax.set_xticks([])
            ax.set_yticks([])

end_time = time.perf_counter()
plt.tight_layout()
plt.show()

elapsed = end_time - start_time
print(f"Segmentadas {NUM_IMAGES} imágenes en {elapsed:.4f} s ({NUM_IMAGES / elapsed:.2f} FPS)")
